# Cahn--Hilliard 2D: competitive F-FNO, CNO, and U-Net baselines

This notebook evaluates three independently trained baselines on the exact
Cahn--Hilliard split and physical-L2 metric used by the AWFNO benchmark:

1. **F-FNO2D-AR** -- FourierFlow-style factorized Fourier neural operator,
   trained on adjacent pairs and recursively rolled out from the true initial
   field only.
2. **CNO2D-Direct(t)** -- the official simplified 2D CNO backbone, supplied
   with the initial field and the known query time. Every future field is
   predicted directly from the initial condition; no predicted or exact
   intermediate field is supplied.
3. **U-Net2D-AR** -- a strong four-level 2D U-Net, trained on adjacent pairs
   and recursively rolled out from the true initial field only.

The split is generated with NumPy `default_rng(2026)` from the 520 stored
trajectories: 400 train-pool, 30 validation, and 70 common test trajectories.
For every sample size, the first N trajectories of the fixed training pool
are used. The test set is never used for fitting, hyperparameter selection,
early stopping, or checkpoint selection.

The reported full-trajectory error is the absolute physical
\(\lVert e\rVert_{L^2(\Omega\times[0,T])}\), using the same spatial rectangle
rule and trapezoidal time rule as the AWFNO benchmark. The exact initial
field is included in every reconstructed trajectory and therefore has zero
error at \(t=0\).



In [1]:
from google.colab import drive
drive.mount("/content/gdrive")



Mounted at /content/gdrive


## Configuration

`cosine_epoch` is a deterministic schedule fixed before training; it does not
react to validation or test results. To use constant learning rates, set
`LR_SCHEDULE = "constant"` before training and keep that choice fixed for all
sample sizes.

The primary F-FNO configuration is FourierFlow's competitive Markov model:
width 64, 24 layers, 16 modes, factor 4, shared Fourier multipliers, weight
normalization, and normalized Gaussian input noise. If runtime is limited,
`FFNO_LAYERS=4` can be used only as an explicitly labelled smaller control;
do not report it as the published 24-layer configuration.



In [2]:
from pathlib import Path

CONFIG = {
    # Exact data artifact used by the AWFNO Cahn--Hilliard benchmark.
    "DATA_PATH": "/content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt",
    "OUTPUT_ROOT": "/content/gdrive/MyDrive/anus/cahn_hilliard_2d_competitive_baselines",

    # Fixed split and sample-size sweep.
    "TRAIN_POOL_SIZE": 400,
    "N_VAL": 30,
    "N_TEST": 70,
    "SPLIT_SEED": 2026,
    "SAMPLE_SIZES": [25, 50, 100, 200, 400],
    "RUN_SAMPLE_SIZES": [25, 50, 100, 200, 400],
    "MODELS_TO_RUN": ["FFNO", "CNO", "UNET"],

    # Geometry and evaluation.
    "DT": 0.01,
    "USE_TRAPEZOIDAL_TIME_L2": True,
    "INCLUDE_T0_IN_FULL_L2": True,
    "ENFORCE_MASS": True,

    # Common optimization and execution.
    "EPOCHS": 200,
    "BATCH_SIZE": 32,
    "EVAL_BATCH_SIZE": 64,
    "VAL_ROLLOUT_INTERVAL": 10,
    "CHECKPOINT_INTERVAL": 10,
    "LR_SCHEDULE": "cosine_epoch",  # "cosine_epoch" or "constant"
    "BASE_SEED": 123400,
    "FORCE_RETRAIN": False,
    "RUN_EXACT_HASH_OVERLAP_CHECK": True,
    "NUM_WORKERS": 0,
    "PIN_MEMORY": True,
    "INFERENCE_REPEATS": 3,
    "GRAD_CLIP_NORM": 1.0,
    "USE_AMP": True,
    "USE_TORCH_COMPILE": False,

    # FourierFlow-style F-FNO2D Markov model.
    "FFNO_WIDTH": 64,
    "FFNO_LAYERS": 24,
    "FFNO_MODES": 16,
    "FFNO_FACTOR": 4,
    "FFNO_FF_LAYERS": 2,
    "FFNO_SHARE_FOURIER_WEIGHT": True,
    "FFNO_WEIGHT_NORM": True,
    "FFNO_FOURIER_INIT_GAIN": 0.1,
    "FFNO_NOISE_STD": 0.01,
    "FFNO_LR": 2.5e-3,
    "FFNO_WEIGHT_DECAY": 1.0e-4,

    # Official simplified CNO2d backbone, adapted only by adding x, y, and
    # query-time channels so it can return u(t) directly from u0.
    "CNO_LAYERS": 3,
    "CNO_CHANNEL_MULTIPLIER": 32,
    "CNO_RES_BLOCKS": 3,
    "CNO_NECK_BLOCKS": 3,
    "CNO_USE_BATCH_NORM": True,
    "CNO_LR": 1.0e-3,
    "CNO_WEIGHT_DECAY": 1.0e-6,

    # Strong four-level autoregressive U-Net.
    "UNET_BASE_CHANNELS": 32,
    "UNET_LEVELS": 4,
    "UNET_NORM": "group",
    "UNET_PADDING_MODE": "reflect",  # appropriate for homogeneous Neumann BC
    "UNET_LR": 1.0e-3,
    "UNET_WEIGHT_DECAY": 1.0e-6,

    # Outputs.
    "SAVE_ROLLOUTS": False,
    "SAVE_PLOTS": True,
    "SHOW_PLOTS": False,
}

DATA_PATH = Path(CONFIG["DATA_PATH"])
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"])
MODEL_ROOT = OUTPUT_ROOT / "models"
RESULT_ROOT = OUTPUT_ROOT / "results"
PLOT_ROOT = OUTPUT_ROOT / "plots"
for directory in (OUTPUT_ROOT, MODEL_ROOT, RESULT_ROOT, PLOT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

assert CONFIG["TRAIN_POOL_SIZE"] + CONFIG["N_VAL"] + CONFIG["N_TEST"] == 500
assert max(CONFIG["SAMPLE_SIZES"]) == CONFIG["TRAIN_POOL_SIZE"]
assert CONFIG["FFNO_SHARE_FOURIER_WEIGHT"] is True



In [3]:
import copy
import gc
import hashlib
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.nn.utils import weight_norm
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings(
    "ignore",
    message="`torch.nn.utils.weight_norm` is deprecated",
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(CONFIG["USE_AMP"] and DEVICE.type == "cuda")
AMP_DTYPE = torch.float16

print("Device:", DEVICE)
print("Automatic mixed precision:", AMP_ENABLED)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GiB):",
        torch.cuda.get_device_properties(0).total_memory / 2**30,
    )
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


with open(OUTPUT_ROOT / "config.json", "w") as stream:
    json.dump(json_ready(CONFIG), stream, indent=2)



Device: cuda
Automatic mixed precision: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory (GiB): 94.97076416015625


## Load the exact data artifact and reproduce the fixed split



In [4]:
def torch_load_safe(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Edit CONFIG['DATA_PATH'] if needed."
    )

raw_data = torch_load_safe(DATA_PATH, map_location="cpu")
if "X_input" not in raw_data or "Y_output" not in raw_data:
    raise KeyError("Expected dataset keys 'X_input' and 'Y_output'.")

X_input = raw_data["X_input"].float()
Y_output = raw_data["Y_output"].float()
if X_input.ndim != 4 or X_input.shape[-1] != 1:
    raise ValueError(f"Expected X_input=(N,Nx,Ny,1), got {X_input.shape}.")
if Y_output.ndim != 5 or Y_output.shape[-1] != 1:
    raise ValueError(f"Expected Y_output=(N,T,Nx,Ny,1), got {Y_output.shape}.")

X_np = X_input[..., 0].contiguous().numpy().astype(np.float32)
Y_np = (
    Y_output[..., 0]
    .permute(0, 2, 3, 1)
    .contiguous()
    .numpy()
    .astype(np.float32)
)
U_ALL = np.concatenate([X_np[..., None], Y_np], axis=-1).astype(np.float32)
del X_input, Y_output, X_np, Y_np

if "x_grid" in raw_data:
    x_grid = raw_data["x_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "xg" in raw_data:
    x_grid = raw_data["xg"].float().reshape(-1).numpy().astype(np.float32)
else:
    x_grid = np.linspace(0.0, 1.0, U_ALL.shape[1], dtype=np.float32)

if "y_grid" in raw_data:
    y_grid = raw_data["y_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "yg" in raw_data:
    y_grid = raw_data["yg"].float().reshape(-1).numpy().astype(np.float32)
else:
    y_grid = np.linspace(0.0, 1.0, U_ALL.shape[2], dtype=np.float32)

del raw_data
gc.collect()

N_TOTAL, NX, NY, NT = U_ALL.shape
if (NX, NY, NT) != (32, 32, 101):
    raise ValueError(
        f"Expected the saved 32x32x101 artifact, received {U_ALL.shape}."
    )
if N_TOTAL < 500:
    raise ValueError(f"Need at least 500 trajectories, received {N_TOTAL}.")
if not np.all(np.isfinite(U_ALL)):
    raise ValueError("Dataset contains NaN or Inf.")

DX = float(np.mean(np.diff(x_grid.astype(np.float64))))
DY = float(np.mean(np.diff(y_grid.astype(np.float64))))
DT = float(CONFIG["DT"])
t_grid = np.arange(NT, dtype=np.float32) * DT

rng = np.random.default_rng(CONFIG["SPLIT_SEED"])
permutation = rng.permutation(N_TOTAL)
train_idx = permutation[: CONFIG["TRAIN_POOL_SIZE"]]
val_start = CONFIG["TRAIN_POOL_SIZE"]
val_stop = val_start + CONFIG["N_VAL"]
test_stop = val_stop + CONFIG["N_TEST"]
val_idx = permutation[val_start:val_stop]
test_idx = permutation[val_stop:test_stop]

U_TRAIN_POOL = np.ascontiguousarray(U_ALL[train_idx], dtype=np.float32)
U_VAL = np.ascontiguousarray(U_ALL[val_idx], dtype=np.float32)
U_TEST = np.ascontiguousarray(U_ALL[test_idx], dtype=np.float32)
del U_ALL
gc.collect()

split_path = OUTPUT_ROOT / "split_indices.npz"
if split_path.exists():
    saved_split = np.load(split_path)
    for key, current in (
        ("train_idx", train_idx),
        ("val_idx", val_idx),
        ("test_idx", test_idx),
    ):
        if not np.array_equal(saved_split[key], current):
            raise RuntimeError(
                f"Existing split file disagrees for {key}; use a new OUTPUT_ROOT."
            )
else:
    np.savez(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        split_seed=np.asarray(CONFIG["SPLIT_SEED"]),
    )

print("\nLoaded Cahn--Hilliard benchmark")
print("--------------------------------")
print("data path       :", DATA_PATH)
print("all data shape  :", (N_TOTAL, NX, NY, NT))
print("training pool   :", U_TRAIN_POOL.shape)
print("validation      :", U_VAL.shape)
print("common test     :", U_TEST.shape)
print("x range / dx    :", (float(x_grid.min()), float(x_grid.max())), DX)
print("y range / dy    :", (float(y_grid.min()), float(y_grid.max())), DY)
print("t range / dt    :", (float(t_grid.min()), float(t_grid.max())), DT)
print("train_idx first :", train_idx[:10])
print("val_idx first   :", val_idx[:10])
print("test_idx first  :", test_idx[:10])


def trajectory_hashes(trajectories):
    hashes = set()
    for trajectory in trajectories:
        contiguous = np.ascontiguousarray(trajectory, dtype=np.float32)
        hashes.add(hashlib.sha256(contiguous.view(np.uint8)).hexdigest())
    return hashes


if CONFIG["RUN_EXACT_HASH_OVERLAP_CHECK"]:
    train_hash = trajectory_hashes(U_TRAIN_POOL)
    val_hash = trajectory_hashes(U_VAL)
    test_hash = trajectory_hashes(U_TEST)
    print("\nExact trajectory overlap")
    print("------------------------")
    print("train_pool vs val :", len(train_hash & val_hash))
    print("train_pool vs test:", len(train_hash & test_hash))
    print("val vs test       :", len(val_hash & test_hash))
    assert not (train_hash & val_hash)
    assert not (train_hash & test_hash)
    assert not (val_hash & test_hash)
    del train_hash, val_hash, test_hash




Loaded Cahn--Hilliard benchmark
--------------------------------
data path       : /content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt
all data shape  : (520, 32, 32, 101)
training pool   : (400, 32, 32, 101)
validation      : (30, 32, 32, 101)
common test     : (70, 32, 32, 101)
x range / dx    : (-1.0, 1.0) 0.06451612903225806
y range / dy    : (-1.0, 1.0) 0.06451612903225806
t range / dt    : (0.0, 1.0) 0.01
train_idx first : [253 450 274  42 518 167 391 192  84 145]
val_idx first   : [285 136 466 366 375 317 156 186 358 148]
test_idx first  : [110 237 469 121 234 226 109 164  43 102]

Exact trajectory overlap
------------------------
train_pool vs val : 0
train_pool vs test: 0
val vs test       : 0


## Physical error and mass-conservation utilities



In [5]:
def time_weights(nt):
    weights = np.ones(nt, dtype=np.float64)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = 0.5
        weights[-1] = 0.5
    return weights


def l2_per_timestep_np(prediction, truth):
    if prediction.shape != truth.shape:
        raise ValueError(f"Shape mismatch: {prediction.shape} vs {truth.shape}")
    n_samples = prediction.shape[0]
    output = np.empty((n_samples, prediction.shape[-1]), dtype=np.float64)
    for start in range(0, n_samples, 16):
        stop = min(start + 16, n_samples)
        error = (
            prediction[start:stop].astype(np.float64)
            - truth[start:stop].astype(np.float64)
        )
        output[start:stop] = np.sqrt(
            np.sum(error * error, axis=(1, 2)) * DX * DY + 1.0e-30
        )
    return output


def rollout_metrics(prediction, truth):
    l2_t = l2_per_timestep_np(prediction, truth)
    used = l2_t if CONFIG["INCLUDE_T0_IN_FULL_L2"] else l2_t[:, 1:]
    weights = time_weights(used.shape[1])[None, :]
    full = np.sqrt(
        np.sum(weights * used * used, axis=1) * DT + 1.0e-30
    )
    final = l2_t[:, -1]
    return {
        "full_mean": float(np.mean(full)),
        "full_std": float(np.std(full)),
        "full_median": float(np.median(full)),
        "final_mean": float(np.mean(final)),
        "final_std": float(np.std(final)),
        "final_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "all_full": full,
        "all_final": final,
    }


def physical_field_l2_torch(prediction, truth):
    return torch.sqrt(
        torch.sum((prediction - truth) ** 2, dim=(-2, -1))
        * DX * DY
        + 1.0e-30
    )


def enforce_mass_torch(prediction, reference):
    if not CONFIG["ENFORCE_MASS"]:
        return prediction
    target_mean = reference.mean(dim=(-2, -1), keepdim=True)
    predicted_mean = prediction.mean(dim=(-2, -1), keepdim=True)
    return prediction + target_mean - predicted_mean


def mass_drift_np(prediction):
    mass = prediction.mean(axis=(1, 2))
    return np.max(np.abs(mass - mass[:, :1]), axis=1)



## Datasets and training-only normalization



In [6]:
class AdjacentPairDataset2D(Dataset):
    """Every adjacent pair from training trajectories; no test data."""

    def __init__(self, trajectories):
        self.trajectories = np.asarray(trajectories, dtype=np.float32)
        self.n_steps = self.trajectories.shape[-1] - 1

    def __len__(self):
        return self.trajectories.shape[0] * self.n_steps

    def __getitem__(self, index):
        trajectory = index // self.n_steps
        step = index % self.n_steps
        current = torch.from_numpy(
            self.trajectories[trajectory, :, :, step]
        )
        future = torch.from_numpy(
            self.trajectories[trajectory, :, :, step + 1]
        )
        return current, future


class DirectTimeDataset2D(Dataset):
    """Pairs (u0, known query time) -> u(t), never using intermediate input."""

    def __init__(self, trajectories):
        self.trajectories = np.asarray(trajectories, dtype=np.float32)
        self.n_future = self.trajectories.shape[-1] - 1

    def __len__(self):
        return self.trajectories.shape[0] * self.n_future

    def __getitem__(self, index):
        trajectory = index // self.n_future
        future_index = index % self.n_future + 1
        u0 = torch.from_numpy(self.trajectories[trajectory, :, :, 0])
        target = torch.from_numpy(
            self.trajectories[trajectory, :, :, future_index]
        )
        query_time = torch.tensor(
            future_index / self.n_future, dtype=torch.float32
        )
        return u0, query_time, target


@dataclass
class ScalarNormalizer:
    mean: float
    std: float
    eps: float = 1.0e-6

    def encode(self, tensor):
        return (tensor - self.mean) / (self.std + self.eps)

    def decode(self, tensor):
        return tensor * (self.std + self.eps) + self.mean


def training_normalizer(trajectories):
    return ScalarNormalizer(
        mean=float(np.mean(trajectories, dtype=np.float64)),
        std=float(np.std(trajectories, dtype=np.float64)),
    )


def normalized_grid_channels(x_values, y_values):
    x = torch.tensor(x_values, dtype=torch.float32)
    y = torch.tensor(y_values, dtype=torch.float32)
    x = (x - x.min()) / (x.max() - x.min() + 1.0e-12)
    y = (y - y.min()) / (y.max() - y.min() + 1.0e-12)
    xx, yy = torch.meshgrid(x, y, indexing="ij")
    return torch.stack([xx, yy], dim=0)



## Model 1: FourierFlow-style factorized F-FNO2D

The implementation below follows the official factorized-grid block:
independent one-dimensional Fourier transforms along x and y, summation of
the two transformed fields, a feed-forward backcast, and a residual update.
The two Fourier multipliers are shared across all 24 layers.



In [7]:
class WNLinear(nn.Linear):
    def __init__(self, in_features, out_features, use_weight_norm):
        super().__init__(in_features, out_features, bias=True)
        if use_weight_norm:
            weight_norm(self)


class FeedForward(nn.Module):
    def __init__(
        self,
        dim,
        factor,
        n_layers,
        use_weight_norm,
        dropout=0.0,
    ):
        super().__init__()
        if n_layers < 1:
            raise ValueError("n_layers must be positive.")
        layers = []
        for index in range(n_layers):
            in_dim = dim if index == 0 else dim * factor
            out_dim = dim if index == n_layers - 1 else dim * factor
            layers.append(WNLinear(in_dim, out_dim, use_weight_norm))
            layers.append(nn.Dropout(dropout))
            layers.append(
                nn.ReLU(inplace=True)
                if index < n_layers - 1
                else nn.Identity()
            )
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)


class FactorizedSpectralLayer2D(nn.Module):
    def __init__(
        self,
        width,
        modes,
        factor,
        n_ff_layers,
        use_weight_norm,
    ):
        super().__init__()
        self.width = int(width)
        self.modes = int(modes)
        self.backcast_ff = FeedForward(
            dim=width,
            factor=factor,
            n_layers=n_ff_layers,
            use_weight_norm=use_weight_norm,
            dropout=0.0,
        )

    def forward_fourier(self, x, fourier_weights):
        # x: (B, Nx, Ny, C), matching FourierFlow's grid_2d.py.
        x_channels = x.permute(0, 3, 1, 2).contiguous()
        batch, _, nx, ny = x_channels.shape

        # Transform along y.
        x_ft_y = torch.fft.rfft(x_channels, dim=-1, norm="ortho")
        out_ft_y = torch.zeros(
            batch,
            self.width,
            nx,
            ny // 2 + 1,
            dtype=x_ft_y.dtype,
            device=x.device,
        )
        retained_y = min(self.modes, ny // 2 + 1)
        weight_y = torch.view_as_complex(
            fourier_weights[0][:, :, :retained_y, :].contiguous()
        )
        out_ft_y[:, :, :, :retained_y] = torch.einsum(
            "bixy,ioy->boxy",
            x_ft_y[:, :, :, :retained_y],
            weight_y,
        )
        transformed_y = torch.fft.irfft(
            out_ft_y, n=ny, dim=-1, norm="ortho"
        )

        # Transform along x.
        x_ft_x = torch.fft.rfft(x_channels, dim=-2, norm="ortho")
        out_ft_x = torch.zeros(
            batch,
            self.width,
            nx // 2 + 1,
            ny,
            dtype=x_ft_x.dtype,
            device=x.device,
        )
        retained_x = min(self.modes, nx // 2 + 1)
        weight_x = torch.view_as_complex(
            fourier_weights[1][:, :, :retained_x, :].contiguous()
        )
        out_ft_x[:, :, :retained_x, :] = torch.einsum(
            "bixy,iox->boxy",
            x_ft_x[:, :, :retained_x, :],
            weight_x,
        )
        transformed_x = torch.fft.irfft(
            out_ft_x, n=nx, dim=-2, norm="ortho"
        )

        transformed = transformed_x + transformed_y
        return transformed.permute(0, 2, 3, 1).contiguous()

    def forward(self, x, fourier_weights):
        return self.backcast_ff(self.forward_fourier(x, fourier_weights))


class FFNO2D_AR(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = int(config["FFNO_WIDTH"])
        modes = int(config["FFNO_MODES"])
        self.n_layers = int(config["FFNO_LAYERS"])
        use_weight_norm = bool(config["FFNO_WEIGHT_NORM"])

        coordinates = normalized_grid_channels(x_grid, y_grid)
        self.register_buffer(
            "coordinates",
            coordinates.unsqueeze(0),
            persistent=False,
        )

        # State plus x and y coordinate channels.
        self.in_proj = WNLinear(3, width, use_weight_norm)

        self.fourier_weights = nn.ParameterList()
        for _ in range(2):
            parameter = nn.Parameter(
                torch.empty(width, width, modes, 2)
            )
            nn.init.xavier_normal_(
                parameter,
                gain=float(config["FFNO_FOURIER_INIT_GAIN"]),
            )
            self.fourier_weights.append(parameter)

        self.layers = nn.ModuleList(
            [
                FactorizedSpectralLayer2D(
                    width=width,
                    modes=modes,
                    factor=int(config["FFNO_FACTOR"]),
                    n_ff_layers=int(config["FFNO_FF_LAYERS"]),
                    use_weight_norm=use_weight_norm,
                )
                for _ in range(self.n_layers)
            ]
        )
        self.out = nn.Sequential(
            WNLinear(width, 128, use_weight_norm),
            WNLinear(128, 1, use_weight_norm),
        )

    def forward(self, normalized_state):
        if normalized_state.ndim != 3:
            raise ValueError(
                f"Expected (B,Nx,Ny), received {normalized_state.shape}."
            )
        batch = normalized_state.shape[0]
        coordinates = (
            self.coordinates.expand(batch, -1, -1, -1)
            .permute(0, 2, 3, 1)
        )
        x = torch.cat(
            [normalized_state.unsqueeze(-1), coordinates], dim=-1
        )
        x = self.in_proj(x)

        backcast = None
        for layer in self.layers:
            backcast = layer(x, self.fourier_weights)
            x = x + backcast
        return self.out(backcast).squeeze(-1)



## Model 2: official simplified CNO2d backbone, direct in time

The CNO blocks, interpolation activation, lift/project maps, residual blocks,
encoder, decoder, and skip expansion reproduce the public simplified CNO2d
implementation. The only benchmark adaptation is the input: normalized
\(u_0\), x, y, and the known normalized query time. This produces \(u(t)\)
without autoregressive feedback.



In [8]:
class CNO_LReLU(nn.Module):
    def __init__(self, in_size, out_size):
        super().__init__()
        self.in_size = int(in_size)
        self.out_size = int(out_size)
        self.activation = nn.LeakyReLU()

    def forward(self, x):
        x = F.interpolate(
            x,
            size=(2 * self.in_size, 2 * self.in_size),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )
        x = self.activation(x)
        return F.interpolate(
            x,
            size=(self.out_size, self.out_size),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )


class CNOBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        in_size,
        out_size,
        use_bn=True,
    ):
        super().__init__()
        self.convolution = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
        )
        self.batch_norm = (
            nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()
        )
        self.activation = CNO_LReLU(in_size, out_size)

    def forward(self, x):
        return self.activation(self.batch_norm(self.convolution(x)))


class CNOLiftProjectBlock(nn.Module):
    def __init__(self, in_channels, out_channels, size, latent_dim=64):
        super().__init__()
        self.intermediate = CNOBlock(
            in_channels,
            latent_dim,
            in_size=size,
            out_size=size,
            use_bn=False,
        )
        self.convolution = nn.Conv2d(
            latent_dim,
            out_channels,
            kernel_size=3,
            padding=1,
        )

    def forward(self, x):
        return self.convolution(self.intermediate(x))


class CNOResidualBlock(nn.Module):
    def __init__(self, channels, size, use_bn=True):
        super().__init__()
        self.convolution1 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1
        )
        self.convolution2 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1
        )
        self.batch_norm1 = (
            nn.BatchNorm2d(channels) if use_bn else nn.Identity()
        )
        self.batch_norm2 = (
            nn.BatchNorm2d(channels) if use_bn else nn.Identity()
        )
        self.activation = CNO_LReLU(size, size)

    def forward(self, x):
        residual = self.convolution1(x)
        residual = self.batch_norm1(residual)
        residual = self.activation(residual)
        residual = self.convolution2(residual)
        residual = self.batch_norm2(residual)
        return x + residual


class CNOResNet(nn.Module):
    def __init__(self, channels, size, num_blocks, use_bn=True):
        super().__init__()
        self.blocks = nn.Sequential(
            *[
                CNOResidualBlock(channels, size, use_bn)
                for _ in range(int(num_blocks))
            ]
        )

    def forward(self, x):
        return self.blocks(x)


class CNO2D_DirectTime(nn.Module):
    def __init__(self, config):
        super().__init__()
        n_layers = int(config["CNO_LAYERS"])
        channel_multiplier = int(config["CNO_CHANNEL_MULTIPLIER"])
        n_res = int(config["CNO_RES_BLOCKS"])
        n_res_neck = int(config["CNO_NECK_BLOCKS"])
        use_bn = bool(config["CNO_USE_BATCH_NORM"])
        size = NX
        if NX != NY or size % (2**n_layers) != 0:
            raise ValueError("CNO expects a divisible square grid.")

        self.n_layers = n_layers
        self.register_buffer(
            "coordinates",
            normalized_grid_channels(x_grid, y_grid).unsqueeze(0),
            persistent=False,
        )

        lift_dim = channel_multiplier // 2
        encoder_features = [lift_dim]
        for index in range(n_layers):
            encoder_features.append(
                (2**index) * channel_multiplier
            )
        decoder_features_in = list(reversed(encoder_features[1:]))
        decoder_features_out = list(reversed(encoder_features[:-1]))
        for index in range(1, n_layers):
            decoder_features_in[index] *= 2

        encoder_sizes = [size // (2**i) for i in range(n_layers + 1)]
        decoder_sizes = [
            size // (2 ** (n_layers - i))
            for i in range(n_layers + 1)
        ]

        # u0, x, y, query time.
        self.lift = CNOLiftProjectBlock(4, lift_dim, size=size)
        self.project = CNOLiftProjectBlock(
            lift_dim + decoder_features_out[-1],
            1,
            size=size,
        )

        self.encoder = nn.ModuleList(
            [
                CNOBlock(
                    encoder_features[i],
                    encoder_features[i + 1],
                    encoder_sizes[i],
                    encoder_sizes[i + 1],
                    use_bn,
                )
                for i in range(n_layers)
            ]
        )
        self.skip_expansion = nn.ModuleList(
            [
                CNOBlock(
                    encoder_features[i],
                    encoder_features[i],
                    encoder_sizes[i],
                    decoder_sizes[n_layers - i],
                    use_bn,
                )
                for i in range(n_layers + 1)
            ]
        )
        self.decoder = nn.ModuleList(
            [
                CNOBlock(
                    decoder_features_in[i],
                    decoder_features_out[i],
                    decoder_sizes[i],
                    decoder_sizes[i + 1],
                    use_bn,
                )
                for i in range(n_layers)
            ]
        )
        self.res_nets = nn.ModuleList(
            [
                CNOResNet(
                    encoder_features[level],
                    encoder_sizes[level],
                    n_res,
                    use_bn,
                )
                for level in range(n_layers)
            ]
        )
        self.neck = CNOResNet(
            encoder_features[n_layers],
            encoder_sizes[n_layers],
            n_res_neck,
            use_bn,
        )

    def forward(self, normalized_u0, query_time):
        if normalized_u0.ndim != 3:
            raise ValueError(normalized_u0.shape)
        batch = normalized_u0.shape[0]
        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        time_channel = query_time.reshape(batch, 1, 1, 1).expand(
            -1, 1, NX, NY
        )
        x = torch.cat(
            [normalized_u0.unsqueeze(1), coordinates, time_channel], dim=1
        )
        x = self.lift(x)
        skip = []
        for index in range(self.n_layers):
            skip_value = self.res_nets[index](x)
            skip.append(skip_value)
            # This follows the official simplified implementation exactly:
            # the down block acts on x; the ResNet value forms the skip.
            x = self.encoder[index](x)
        x = self.neck(x)

        for index in range(self.n_layers):
            if index == 0:
                x = self.skip_expansion[self.n_layers](x)
            else:
                expanded_skip = self.skip_expansion[
                    self.n_layers - index
                ](skip[-index])
                x = torch.cat([x, expanded_skip], dim=1)
            x = self.decoder[index](x)

        x = torch.cat([x, self.skip_expansion[0](skip[0])], dim=1)
        return self.project(x).squeeze(1)



## Model 3: strong autoregressive 2D U-Net

Reflection padding is used because the benchmark has homogeneous Neumann
boundary conditions. Group normalization avoids unstable batch statistics in
the small-data settings. This is labelled as `U-Net2D-AR-Neumann` in every
result file.



In [9]:
def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class UNetConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, config):
        super().__init__()
        padding_mode = config["UNET_PADDING_MODE"]
        norm_kind = config["UNET_NORM"]

        def normalization(channels):
            if norm_kind == "group":
                return nn.GroupNorm(group_count(channels), channels)
            if norm_kind == "batch":
                return nn.BatchNorm2d(channels)
            if norm_kind == "none":
                return nn.Identity()
            raise ValueError(norm_kind)

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=padding_mode,
            ),
            normalization(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=padding_mode,
            ),
            normalization(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UNet2D_AR(nn.Module):
    def __init__(self, config):
        super().__init__()
        base = int(config["UNET_BASE_CHANNELS"])
        levels = int(config["UNET_LEVELS"])
        if levels != 4:
            raise ValueError("This audited implementation uses four levels.")
        channels = [base, 2 * base, 4 * base, 8 * base, 16 * base]
        self.register_buffer(
            "coordinates",
            normalized_grid_channels(x_grid, y_grid).unsqueeze(0),
            persistent=False,
        )

        self.encoders = nn.ModuleList(
            [
                UNetConvBlock(3, channels[0], config),
                UNetConvBlock(channels[0], channels[1], config),
                UNetConvBlock(channels[1], channels[2], config),
                UNetConvBlock(channels[2], channels[3], config),
            ]
        )
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = UNetConvBlock(
            channels[3], channels[4], config
        )
        self.upconvs = nn.ModuleList(
            [
                nn.ConvTranspose2d(
                    channels[4], channels[3], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[3], channels[2], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[2], channels[1], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[1], channels[0], kernel_size=2, stride=2
                ),
            ]
        )
        self.decoders = nn.ModuleList(
            [
                UNetConvBlock(2 * channels[3], channels[3], config),
                UNetConvBlock(2 * channels[2], channels[2], config),
                UNetConvBlock(2 * channels[1], channels[1], config),
                UNetConvBlock(2 * channels[0], channels[0], config),
            ]
        )
        self.output = nn.Conv2d(channels[0], 1, kernel_size=1)

    def forward(self, normalized_state):
        batch = normalized_state.shape[0]
        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        x = torch.cat([normalized_state.unsqueeze(1), coordinates], dim=1)
        skip = []
        for encoder in self.encoders:
            x = encoder(x)
            skip.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for index, (upconv, decoder) in enumerate(
            zip(self.upconvs, self.decoders)
        ):
            x = upconv(x)
            x = torch.cat([x, skip[-(index + 1)]], dim=1)
            x = decoder(x)
        return self.output(x).squeeze(1)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def make_model(model_key):
    if model_key == "FFNO":
        return FFNO2D_AR(CONFIG)
    if model_key == "CNO":
        return CNO2D_DirectTime(CONFIG)
    if model_key == "UNET":
        return UNet2D_AR(CONFIG)
    raise ValueError(model_key)


MODEL_LABELS = {
    "FFNO": (
        f"F-FNO2D-AR-W{CONFIG['FFNO_WIDTH']}-"
        f"L{CONFIG['FFNO_LAYERS']}"
    ),
    "CNO": "CNO2D-Direct(t)",
    "UNET": "U-Net2D-AR-Neumann",
}

set_seed(CONFIG["BASE_SEED"])
PARAMETER_COUNTS = {}
for model_key in ("FFNO", "CNO", "UNET"):
    audit_model = make_model(model_key)
    PARAMETER_COUNTS[model_key] = count_parameters(audit_model)
    print(
        f"{MODEL_LABELS[model_key]:28s} parameters: "
        f"{PARAMETER_COUNTS[model_key]:,}"
    )
    del audit_model
gc.collect()



F-FNO2D-AR-W64-L24           parameters: 1,072,834
CNO2D-Direct(t)              parameters: 1,624,177
U-Net2D-AR-Neumann           parameters: 7,765,985


0

## Training, validation-only checkpointing, and full-trajectory prediction



In [12]:
def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def optimizer_hyperparameters(model_key):
    if model_key == "FFNO":
        return CONFIG["FFNO_LR"], CONFIG["FFNO_WEIGHT_DECAY"]
    if model_key == "CNO":
        return CONFIG["CNO_LR"], CONFIG["CNO_WEIGHT_DECAY"]
    if model_key == "UNET":
        return CONFIG["UNET_LR"], CONFIG["UNET_WEIGHT_DECAY"]
    raise ValueError(model_key)


def make_scheduler(optimizer):
    if CONFIG["LR_SCHEDULE"] == "cosine_epoch":
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=CONFIG["EPOCHS"],
            eta_min=0.0,
        )
    if CONFIG["LR_SCHEDULE"] == "constant":
        return None
    raise ValueError(CONFIG["LR_SCHEDULE"])


def amp_enabled_for(model_key):
    return bool(AMP_ENABLED and model_key == "UNET")


def make_grad_scaler(model_key):
    return torch.cuda.amp.GradScaler(
        enabled=amp_enabled_for(model_key)
    )


def build_dataset(model_key, trajectories):
    if model_key == "CNO":
        return DirectTimeDataset2D(trajectories)
    return AdjacentPairDataset2D(trajectories)


def unpack_training_batch(model_key, batch):
    if model_key == "CNO":
        current, query_time, target = batch
        return (
            current.to(DEVICE, non_blocking=True),
            query_time.to(DEVICE, non_blocking=True),
            target.to(DEVICE, non_blocking=True),
        )
    current, target = batch
    return (
        current.to(DEVICE, non_blocking=True),
        None,
        target.to(DEVICE, non_blocking=True),
    )


def model_prediction_physical(
    model_key,
    model,
    current,
    query_time,
    normalizer,
    add_training_noise=False,
):
    normalized_current = normalizer.encode(current)
    if (
        add_training_noise
        and model_key == "FFNO"
        and CONFIG["FFNO_NOISE_STD"] > 0
    ):
        normalized_current = (
            normalized_current
            + float(CONFIG["FFNO_NOISE_STD"])
            * torch.randn_like(normalized_current)
        )

    if model_key == "CNO":
        normalized_prediction = model(
            normalized_current, query_time
        )
    else:
        normalized_prediction = model(normalized_current)

    prediction = normalizer.decode(normalized_prediction)
    prediction = enforce_mass_torch(prediction, current)
    return prediction


def train_one_epoch(
    model_key,
    model,
    loader,
    optimizer,
    scaler,
    normalizer,
):
    model.train()
    loss_sum = 0.0
    l2_sum = 0.0
    count = 0

    for batch in loader:
        current, query_time, target = unpack_training_batch(
            model_key, batch
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=amp_enabled_for(model_key),
        ):
            prediction = model_prediction_physical(
                model_key,
                model,
                current,
                query_time,
                normalizer,
                add_training_noise=True,
            )
            loss = F.mse_loss(
                normalizer.encode(prediction),
                normalizer.encode(target),
            )

        scaler.scale(loss).backward()
        if CONFIG["GRAD_CLIP_NORM"] is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                float(CONFIG["GRAD_CLIP_NORM"]),
            )
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            field_l2 = physical_field_l2_torch(prediction, target)
        batch_size = current.shape[0]
        loss_sum += float(loss.detach().item()) * batch_size
        l2_sum += float(field_l2.sum().item())
        count += batch_size

    return loss_sum / count, l2_sum / count


@torch.no_grad()
def evaluate_pairs(model_key, model, loader, normalizer):
    model.eval()
    loss_sum = 0.0
    l2_sum = 0.0
    count = 0
    for batch in loader:
        current, query_time, target = unpack_training_batch(
            model_key, batch
        )
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=amp_enabled_for(model_key),
        ):
            prediction = model_prediction_physical(
                model_key,
                model,
                current,
                query_time,
                normalizer,
                add_training_noise=False,
            )
            loss = F.mse_loss(
                normalizer.encode(prediction),
                normalizer.encode(target),
            )
        field_l2 = physical_field_l2_torch(prediction, target)
        batch_size = current.shape[0]
        loss_sum += float(loss.item()) * batch_size
        l2_sum += float(field_l2.sum().item())
        count += batch_size
    return loss_sum / count, l2_sum / count


@torch.no_grad()
def recursive_trajectory(model, initial_states, normalizer):
    """Autoregressive trajectory using only u0 and preceding predictions."""
    model.eval()
    n_samples = initial_states.shape[0]
    prediction = np.empty(
        (n_samples, NX, NY, NT), dtype=np.float32
    )
    prediction[:, :, :, 0] = initial_states

    chunk_size = int(CONFIG["EVAL_BATCH_SIZE"])
    for start in range(0, n_samples, chunk_size):
        stop = min(start + chunk_size, n_samples)
        current = torch.as_tensor(
            initial_states[start:stop],
            dtype=torch.float32,
            device=DEVICE,
        )
        recursive_model_key = (
            "FFNO" if isinstance(model, FFNO2D_AR) else "UNET"
        )
        for step in range(1, NT):
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=amp_enabled_for(recursive_model_key),
            ):
                future = model_prediction_physical(
                    recursive_model_key,
                    model,
                    current,
                    None,
                    normalizer,
                    add_training_noise=False,
                )
            current = future.float()
            prediction[start:stop, :, :, step] = (
                current.detach().cpu().numpy()
            )
    return prediction


@torch.no_grad()
def direct_time_trajectory(model, initial_states, normalizer):
    """Each u(t) is queried directly from u0; no recursive input is used."""
    model.eval()
    n_samples = initial_states.shape[0]
    prediction = np.empty(
        (n_samples, NX, NY, NT), dtype=np.float32
    )
    prediction[:, :, :, 0] = initial_states

    # Flatten (sample, future time) queries and evaluate in safe chunks.
    sample_ids = np.repeat(np.arange(n_samples), NT - 1)
    time_ids = np.tile(np.arange(1, NT), n_samples)
    chunk_size = int(CONFIG["EVAL_BATCH_SIZE"])
    for start in range(0, len(sample_ids), chunk_size):
        stop = min(start + chunk_size, len(sample_ids))
        ids = sample_ids[start:stop]
        steps = time_ids[start:stop]
        u0 = torch.as_tensor(
            initial_states[ids],
            dtype=torch.float32,
            device=DEVICE,
        )
        query_time = torch.as_tensor(
            steps / (NT - 1),
            dtype=torch.float32,
            device=DEVICE,
        )
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=amp_enabled_for("CNO"),
        ):
            future = model_prediction_physical(
                "CNO",
                model,
                u0,
                query_time,
                normalizer,
                add_training_noise=False,
            )
        future_np = future.float().cpu().numpy()
        for local_index, (sample_id, step) in enumerate(zip(ids, steps)):
            prediction[sample_id, :, :, step] = future_np[local_index]
    return prediction


def predict_full_trajectory(model_key, model, trajectories, normalizer):
    initial_states = trajectories[:, :, :, 0]
    if model_key == "CNO":
        return direct_time_trajectory(model, initial_states, normalizer)
    return recursive_trajectory(model, initial_states, normalizer)


def validation_full_error(model_key, model, normalizer):
    prediction = predict_full_trajectory(
        model_key, model, U_VAL, normalizer
    )
    value = rollout_metrics(prediction, U_VAL)["full_mean"]
    del prediction
    gc.collect()
    return value


def fit_or_load(model_key, n_train):
    run_seed = (
        CONFIG["BASE_SEED"]
        + 10000 * ("FFNO", "CNO", "UNET").index(model_key)
        + int(n_train)
    )
    run_directory = MODEL_ROOT / model_key / f"N{n_train}"
    run_directory.mkdir(parents=True, exist_ok=True)
    done_path = run_directory / "done.pt"
    best_path = run_directory / "best.pt"
    latest_path = run_directory / "latest.pt"

    if CONFIG["FORCE_RETRAIN"]:
        for path in (done_path, best_path, latest_path):
            if path.exists():
                path.unlink()

    normalizer = training_normalizer(U_TRAIN_POOL[:n_train])
    model = make_model(model_key).to(DEVICE)
    if count_parameters(model) != PARAMETER_COUNTS[model_key]:
        raise RuntimeError("Parameter count changed unexpectedly.")

    if done_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        print(f"Loading completed checkpoint: {done_path}")
        checkpoint = torch_load_safe(done_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        restored_normalizer = ScalarNormalizer(
            mean=float(checkpoint["normalizer"]["mean"]),
            std=float(checkpoint["normalizer"]["std"]),
        )
        return {
            "model": model,
            "normalizer": restored_normalizer,
            "history": checkpoint["history"],
            "best_epoch": int(checkpoint["best_epoch"]),
            "best_val_full": float(checkpoint["best_val_full"]),
            "train_time_sec": float(checkpoint["train_time_sec"]),
            "checkpoint": str(done_path),
        }

    set_seed(run_seed)
    model = make_model(model_key).to(DEVICE)
    if CONFIG["USE_TORCH_COMPILE"]:
        model = torch.compile(model)

    learning_rate, weight_decay = optimizer_hyperparameters(model_key)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(learning_rate),
        weight_decay=float(weight_decay),
    )
    scheduler = make_scheduler(optimizer)
    scaler = make_grad_scaler(model_key)

    train_dataset = build_dataset(
        model_key, U_TRAIN_POOL[:n_train]
    )
    validation_dataset = build_dataset(model_key, U_VAL)
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(run_seed + 100000)
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(CONFIG["BATCH_SIZE"]),
        shuffle=True,
        generator=shuffle_generator,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(
            CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"
        ),
        drop_last=False,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=int(CONFIG["EVAL_BATCH_SIZE"]),
        shuffle=False,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(
            CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"
        ),
        drop_last=False,
    )

    history = {
        "epoch": [],
        "lr": [],
        "train_loss": [],
        "train_field_l2": [],
        "val_pair_loss": [],
        "val_field_l2": [],
        "val_full_l2": [],
    }
    best_epoch = -1
    best_val_full = float("inf")
    start_epoch = 1
    elapsed_before = 0.0

    if latest_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        latest = torch_load_safe(latest_path, map_location=DEVICE)
        if latest["model_key"] != model_key:
            raise RuntimeError("Checkpoint model mismatch.")
        if int(latest["n_train"]) != int(n_train):
            raise RuntimeError("Checkpoint sample-size mismatch.")
        model.load_state_dict(latest["model_state"])
        optimizer.load_state_dict(latest["optimizer_state"])
        optimizer_to_device(optimizer, DEVICE)
        if scheduler is not None and latest["scheduler_state"] is not None:
            scheduler.load_state_dict(latest["scheduler_state"])
        scaler.load_state_dict(latest["scaler_state"])
        history = latest["history"]
        best_epoch = int(latest["best_epoch"])
        best_val_full = float(latest["best_val_full"])
        start_epoch = int(latest["epoch"]) + 1
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        if "shuffle_generator_state" in latest:
            shuffle_generator.set_state(latest["shuffle_generator_state"])
        print(
            f"Resuming {MODEL_LABELS[model_key]} N={n_train} "
            f"at epoch {start_epoch}."
        )

    print("\n" + "=" * 100)
    print(f"{MODEL_LABELS[model_key]} | N_train={n_train}")
    print("=" * 100)
    print("training trajectories :", U_TRAIN_POOL[:n_train].shape)
    print("training examples     :", len(train_dataset))
    print("validation examples   :", len(validation_dataset))
    print("parameters            :", f"{count_parameters(model):,}")
    print("batch size            :", CONFIG["BATCH_SIZE"])
    print("learning rate         :", learning_rate)
    print("weight decay          :", weight_decay)
    print("learning-rate schedule:", CONFIG["LR_SCHEDULE"])
    print("mass enforcement      :", CONFIG["ENFORCE_MASS"])
    print("test used for selection: no")

    segment_start = time.perf_counter()
    for epoch in range(start_epoch, int(CONFIG["EPOCHS"]) + 1):
        train_loss, train_l2 = train_one_epoch(
            model_key,
            model,
            train_loader,
            optimizer,
            scaler,
            normalizer,
        )
        val_loss, val_l2 = evaluate_pairs(
            model_key, model, validation_loader, normalizer
        )

        run_rollout = (
            epoch % int(CONFIG["VAL_ROLLOUT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        )
        val_full = (
            validation_full_error(model_key, model, normalizer)
            if run_rollout
            else float("nan")
        )

        if scheduler is not None:
            scheduler.step()
        current_lr = float(optimizer.param_groups[0]["lr"])
        history["epoch"].append(epoch)
        history["lr"].append(current_lr)
        history["train_loss"].append(train_loss)
        history["train_field_l2"].append(train_l2)
        history["val_pair_loss"].append(val_loss)
        history["val_field_l2"].append(val_l2)
        history["val_full_l2"].append(val_full)

        if run_rollout and np.isfinite(val_full) and val_full < best_val_full:
            best_epoch = int(epoch)
            best_val_full = float(val_full)
            torch.save(
                {
                    "model_key": model_key,
                    "n_train": int(n_train),
                    "model_state": state_dict_to_cpu(model),
                    "epoch": best_epoch,
                    "best_val_full": best_val_full,
                    "normalizer": {
                        "mean": normalizer.mean,
                        "std": normalizer.std,
                    },
                    "config": copy.deepcopy(CONFIG),
                },
                best_path,
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"[{model_key}] {epoch:04d}/{CONFIG['EPOCHS']} | "
                f"train loss {train_loss:.6e} | "
                f"train field L2 {train_l2:.6e} | "
                f"val pair loss {val_loss:.6e} | "
                f"val field L2 {val_l2:.6e} | "
                f"val full L2 {val_full:.6e} | "
                f"lr {current_lr:.3e}"
            )

        if (
            epoch % int(CONFIG["CHECKPOINT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        ):
            elapsed = elapsed_before + (
                time.perf_counter() - segment_start
            )
            torch.save(
                {
                    "model_key": model_key,
                    "n_train": int(n_train),
                    "epoch": int(epoch),
                    "model_state": state_dict_to_cpu(model),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": (
                        scheduler.state_dict()
                        if scheduler is not None
                        else None
                    ),
                    "scaler_state": scaler.state_dict(),
                    "history": history,
                    "best_epoch": int(best_epoch),
                    "best_val_full": float(best_val_full),
                    "normalizer": {
                        "mean": normalizer.mean,
                        "std": normalizer.std,
                    },
                    "train_time_sec": float(elapsed),
                    "shuffle_generator_state": (
                        shuffle_generator.get_state()
                    ),
                    "config": copy.deepcopy(CONFIG),
                },
                latest_path,
            )

    total_train_time = elapsed_before + (
        time.perf_counter() - segment_start
    )
    if not best_path.exists():
        raise RuntimeError(
            f"No finite validation checkpoint for {model_key}, N={n_train}."
        )

    best = torch_load_safe(best_path, map_location=DEVICE)
    model.load_state_dict(best["model_state"])
    best_epoch = int(best["epoch"])
    best_val_full = float(best["best_val_full"])
    done_bundle = {
        "model_key": model_key,
        "n_train": int(n_train),
        "model_state": state_dict_to_cpu(model),
        "history": history,
        "best_epoch": best_epoch,
        "best_val_full": best_val_full,
        "train_time_sec": float(total_train_time),
        "normalizer": {
            "mean": normalizer.mean,
            "std": normalizer.std,
        },
        "parameters": int(count_parameters(model)),
        "config": copy.deepcopy(CONFIG),
    }
    torch.save(done_bundle, done_path)
    print(
        f"Selected epoch {best_epoch} with validation full L2 "
        f"{best_val_full:.6e}"
    )
    return {
        "model": model,
        "normalizer": normalizer,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_full": best_val_full,
        "train_time_sec": float(total_train_time),
        "checkpoint": str(done_path),
    }


def timed_test_prediction(model_key, model, normalizer):
    # Warm-up outside the timed region.
    warm_trajectories = U_TEST[: min(2, len(U_TEST))]
    _ = predict_full_trajectory(
        model_key, model, warm_trajectories, normalizer
    )
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    prediction = None
    timings = []
    for _ in range(int(CONFIG["INFERENCE_REPEATS"])):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = predict_full_trajectory(
            model_key, model, U_TEST, normalizer
        )
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    return prediction, float(np.median(timings)), timings



## Diagnostics, sample-size sweep, and paper-ready tables



In [13]:
def plot_training_history(history, model_key, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    epochs = np.asarray(history["epoch"], dtype=int)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    axes[0].semilogy(
        epochs,
        history["train_field_l2"],
        label="train field L2",
    )
    axes[0].semilogy(
        epochs,
        history["val_field_l2"],
        label="validation field L2",
    )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("absolute physical field L2")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    val_full = np.asarray(history["val_full_l2"], dtype=float)
    finite = np.isfinite(val_full)
    axes[1].semilogy(
        epochs[finite],
        val_full[finite],
        marker="o",
        label="validation full-trajectory L2",
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("absolute physical trajectory L2")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    figure.suptitle(f"{MODEL_LABELS[model_key]}, N={n_train}")
    figure.tight_layout()
    path = (
        PLOT_ROOT
        / f"N{n_train}_{model_key.lower()}_training_diagnostics.png"
    )
    figure.savefig(path, dpi=170)
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)


def plot_l2_per_time(metrics, model_key, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], linewidth=2, label="mean")
    axis.plot(
        t_grid,
        metrics["median_t"],
        "--",
        linewidth=2,
        label="median",
    )
    axis.fill_between(
        t_grid,
        np.maximum(metrics["mean_t"] - metrics["std_t"], 0.0),
        metrics["mean_t"] + metrics["std_t"],
        alpha=0.2,
        label="mean ± std",
    )
    axis.set_xlabel("time")
    axis.set_ylabel("absolute physical L2")
    axis.set_title(f"{MODEL_LABELS[model_key]}, N={n_train}")
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    path = (
        PLOT_ROOT
        / f"N{n_train}_{model_key.lower()}_l2_per_time.png"
    )
    figure.savefig(path, dpi=170)
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)


def plot_solution_panel(prediction, model_key, n_train, sample=0):
    if not CONFIG["SAVE_PLOTS"]:
        return
    time_indices = [0, 10, 25, 50, 100]
    figure, axes = plt.subplots(
        3,
        len(time_indices),
        figsize=(3.0 * len(time_indices), 8.0),
        constrained_layout=True,
    )
    truth = U_TEST[sample]
    error_limit = 0.0
    for time_index in time_indices:
        error_limit = max(
            error_limit,
            float(
                np.max(
                    np.abs(
                        prediction[sample, :, :, time_index]
                        - truth[:, :, time_index]
                    )
                )
            ),
        )
    field_limit = max(
        float(np.max(np.abs(truth))),
        float(np.max(np.abs(prediction[sample]))),
    )
    for column, time_index in enumerate(time_indices):
        exact = truth[:, :, time_index]
        predicted = prediction[sample, :, :, time_index]
        error = predicted - exact
        axes[0, column].imshow(
            exact.T,
            origin="lower",
            cmap="coolwarm",
            vmin=-field_limit,
            vmax=field_limit,
        )
        axes[1, column].imshow(
            predicted.T,
            origin="lower",
            cmap="coolwarm",
            vmin=-field_limit,
            vmax=field_limit,
        )
        axes[2, column].imshow(
            error.T,
            origin="lower",
            cmap="RdBu_r",
            vmin=-error_limit,
            vmax=error_limit,
        )
        axes[0, column].set_title(f"t={t_grid[time_index]:.2f}")
        for row in range(3):
            axes[row, column].set_xticks([])
            axes[row, column].set_yticks([])
    axes[0, 0].set_ylabel("reference")
    axes[1, 0].set_ylabel("prediction")
    axes[2, 0].set_ylabel("error")
    figure.suptitle(
        f"{MODEL_LABELS[model_key]}, N={n_train}, test sample {sample}"
    )
    path = (
        PLOT_ROOT
        / f"N{n_train}_{model_key.lower()}_solution_panel.png"
    )
    figure.savefig(path, dpi=170)
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)


all_results = []
for model_key in CONFIG["MODELS_TO_RUN"]:
    if model_key not in MODEL_LABELS:
        raise ValueError(f"Unknown model key {model_key}.")
    for n_train in CONFIG["RUN_SAMPLE_SIZES"]:
        if n_train not in CONFIG["SAMPLE_SIZES"]:
            raise ValueError(f"Unexpected sample size {n_train}.")

        fitted = fit_or_load(model_key, n_train)
        prediction, inference_median, inference_all = timed_test_prediction(
            model_key,
            fitted["model"],
            fitted["normalizer"],
        )
        metrics = rollout_metrics(prediction, U_TEST)
        mass_drift = mass_drift_np(prediction)
        result = {
            "n_train": int(n_train),
            "model_key": model_key,
            "model": MODEL_LABELS[model_key],
            "protocol": (
                "direct_u0_and_query_time"
                if model_key == "CNO"
                else "one_step_autoregressive"
            ),
            "parameters": int(count_parameters(fitted["model"])),
            "best_epoch": int(fitted["best_epoch"]),
            "validation_full_l2": float(fitted["best_val_full"]),
            "full_mean": metrics["full_mean"],
            "full_std": metrics["full_std"],
            "full_median": metrics["full_median"],
            "final_mean": metrics["final_mean"],
            "final_std": metrics["final_std"],
            "final_median": metrics["final_median"],
            "mass_drift_mean": float(np.mean(mass_drift)),
            "mass_drift_max": float(np.max(mass_drift)),
            "train_time_sec": float(fitted["train_time_sec"]),
            "inference_time_median_sec": inference_median,
            "inference_time_all_sec": inference_all,
            "checkpoint": fitted["checkpoint"],
        }
        all_results.append(result)

        np.savez_compressed(
            RESULT_ROOT / f"N{n_train}_{model_key.lower()}_metrics.npz",
            mean_t=metrics["mean_t"],
            std_t=metrics["std_t"],
            median_t=metrics["median_t"],
            all_full=metrics["all_full"],
            all_final=metrics["all_final"],
            mass_drift=mass_drift,
            inference_times=np.asarray(inference_all),
        )
        if CONFIG["SAVE_ROLLOUTS"]:
            np.save(
                RESULT_ROOT
                / f"N{n_train}_{model_key.lower()}_rollout.npy",
                prediction.astype(np.float32),
            )

        plot_training_history(
            fitted["history"], model_key, n_train
        )
        plot_l2_per_time(metrics, model_key, n_train)
        plot_solution_panel(prediction, model_key, n_train, sample=0)

        print("\nTest result")
        print("-----------")
        print("model             :", MODEL_LABELS[model_key])
        print("N_train           :", n_train)
        print("parameters        :", f"{result['parameters']:,}")
        print(
            "full mean/std     :",
            f"{result['full_mean']:.6e}",
            "/",
            f"{result['full_std']:.6e}",
        )
        print("full median       :", f"{result['full_median']:.6e}")
        print(
            "final mean/std    :",
            f"{result['final_mean']:.6e}",
            "/",
            f"{result['final_std']:.6e}",
        )
        print("final median      :", f"{result['final_median']:.6e}")
        print("mean mass drift   :", f"{result['mass_drift_mean']:.6e}")
        print("training time     :", f"{result['train_time_sec']:.2f} s")
        print(
            "inference median  :",
            f"{result['inference_time_median_sec']:.4f} s",
        )

        del fitted, prediction, metrics, mass_drift
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()



Loading completed checkpoint: /content/gdrive/MyDrive/anus/cahn_hilliard_2d_competitive_baselines/models/FFNO/N25/done.pt

Test result
-----------
model             : F-FNO2D-AR-W64-L24
N_train           : 25
parameters        : 1,072,834
full mean/std     : 9.521558e-01 / 1.508856e-01
full median       : 9.415392e-01
final mean/std    : 1.391968e+00 / 2.682670e-01
final median      : 1.395491e+00
mean mass drift   : 3.252828e-07
training time     : 484.17 s
inference median  : 1.7529 s
Loading completed checkpoint: /content/gdrive/MyDrive/anus/cahn_hilliard_2d_competitive_baselines/models/FFNO/N50/done.pt

Test result
-----------
model             : F-FNO2D-AR-W64-L24
N_train           : 50
parameters        : 1,072,834
full mean/std     : 1.502144e+00 / 1.196921e-01
full median       : 1.509277e+00
final mean/std    : 2.005656e+00 / 1.976120e-01
final median      : 2.042011e+00
mean mass drift   : 1.776404e-07
training time     : 831.61 s
inference median  : 1.7457 s


/tmp/ipykernel_3090/1902231645.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  return torch.cuda.amp.GradScaler(


TypeError: RNG state must be a torch.ByteTensor

In [ ]:
new_csv = RESULT_ROOT / "cahn_hilliard_new_baselines_summary.csv"
new_json = RESULT_ROOT / "cahn_hilliard_new_baselines_summary.json"
current_results = pd.DataFrame(all_results)
if new_csv.exists():
    previous_results = pd.read_csv(new_csv)
    new_results = pd.concat(
        [previous_results, current_results],
        ignore_index=True,
    )
    new_results = new_results.drop_duplicates(
        subset=["n_train", "model_key"],
        keep="last",
    )
else:
    new_results = current_results
new_results = new_results.sort_values(["n_train", "model_key"])
new_results.to_csv(new_csv, index=False)
with open(new_json, "w") as stream:
    json.dump(
        json_ready(new_results.to_dict(orient="records")),
        stream,
        indent=2,
    )

print("\n" + "=" * 132)
print("CAHN--HILLIARD 2D NEW BASELINES: ABSOLUTE PHYSICAL L2")
print("=" * 132)
for row in all_results:
    print(
        f"N={row['n_train']:4d} | "
        f"{row['model']:28s} | "
        f"full mean {row['full_mean']:.6e} | "
        f"std {row['full_std']:.6e} | "
        f"median {row['full_median']:.6e} | "
        f"final mean {row['final_mean']:.6e} | "
        f"final median {row['final_median']:.6e} | "
        f"params {row['parameters']:,} | "
        f"train {row['train_time_sec']:.2f}s | "
        f"inference {row['inference_time_median_sec']:.4f}s"
    )
print("=" * 132)
print("Saved CSV :", new_csv)
print("Saved JSON:", new_json)



## Combine with the existing manuscript values

The following values are transcribed from the current manuscript table; they
are not recomputed in this notebook. Keeping them separate from
`cahn_hilliard_new_baselines_summary.csv` prevents accidental confusion
between newly measured and transcribed numbers.



In [ ]:
MANUSCRIPT_REFERENCE = {
    "TB-AWFNO": {
        "full": [0.852, 0.726, 0.628, 0.618, 0.598],
        "final": [1.286, 1.115, 0.973, 0.964, 0.928],
    },
    "AWFNO": {
        "full": [0.913, 0.777, 0.662, 0.624, 0.603],
        "final": [1.347, 1.186, 1.023, 0.966, 0.940],
    },
    "WFNO": {
        "full": [1.119, 1.100, 0.930, 0.712, 0.807],
        "final": [1.618, 1.577, 1.357, 1.097, 1.209],
    },
    "FNO": {
        "full": [0.965, 0.947, 0.923, 0.877, 0.831],
        "final": [1.362, 1.347, 1.327, 1.275, 1.226],
    },
}

reference_rows = []
for model_name, values in MANUSCRIPT_REFERENCE.items():
    for index, n_train in enumerate(CONFIG["SAMPLE_SIZES"]):
        reference_rows.append(
            {
                "n_train": n_train,
                "model": model_name,
                "full_mean": values["full"][index],
                "final_mean": values["final"][index],
                "source": "current_manuscript_transcription",
            }
        )

measured_rows = new_results[
    ["n_train", "model", "full_mean", "final_mean"]
].copy()
measured_rows["source"] = "measured_by_this_notebook"
combined_long = pd.concat(
    [pd.DataFrame(reference_rows), measured_rows],
    ignore_index=True,
)
combined_long.to_csv(
    RESULT_ROOT / "cahn_hilliard_combined_long.csv",
    index=False,
)

full_table = combined_long.pivot(
    index="n_train", columns="model", values="full_mean"
).sort_index()
final_table = combined_long.pivot(
    index="n_train", columns="model", values="final_mean"
).sort_index()
full_table.to_csv(RESULT_ROOT / "paper_table_full_l2.csv")
final_table.to_csv(RESULT_ROOT / "paper_table_final_l2.csv")
with open(RESULT_ROOT / "paper_table_full_l2.tex", "w") as stream:
    stream.write(
        full_table.to_latex(
            float_format=lambda value: f"{value:.3f}",
            na_rep="--",
        )
    )
with open(RESULT_ROOT / "paper_table_final_l2.tex", "w") as stream:
    stream.write(
        final_table.to_latex(
            float_format=lambda value: f"{value:.3f}",
            na_rep="--",
        )
    )

print("\nMean full-trajectory absolute physical L2")
display(full_table)
print("\nMean final-time absolute physical L2")
display(final_table)

if CONFIG["SAVE_PLOTS"]:
    for metric_name, table, ylabel in (
        (
            "full",
            full_table,
            r"mean absolute $L^2(\Omega\times[0,T])$",
        ),
        ("final", final_table, r"mean absolute final-time $L^2(\Omega)$"),
    ):
        figure, axis = plt.subplots(figsize=(9.0, 5.3))
        for column in table.columns:
            axis.loglog(
                table.index,
                table[column],
                marker="o",
                linewidth=2,
                label=column,
            )
        axis.set_xlabel("training trajectories")
        axis.set_ylabel(ylabel)
        axis.grid(True, which="both", alpha=0.3)
        axis.legend(ncol=2, fontsize=9)
        figure.tight_layout()
        figure.savefig(
            PLOT_ROOT / f"combined_sample_size_{metric_name}_l2.png",
            dpi=180,
        )
        if CONFIG["SHOW_PLOTS"]:
            plt.show()
        plt.close(figure)



## Parameter and protocol table for the rebuttal



In [ ]:
protocol_table = pd.DataFrame(
    [
        {
            "model": MODEL_LABELS["FFNO"],
            "input_at_test": "u0, then preceding prediction",
            "output_per_call": "one next field",
            "rollout": "autoregressive",
            "parameters": PARAMETER_COUNTS["FFNO"],
        },
        {
            "model": MODEL_LABELS["CNO"],
            "input_at_test": "u0 and known query time",
            "output_per_call": "field at query time",
            "rollout": "non-autoregressive direct queries",
            "parameters": PARAMETER_COUNTS["CNO"],
        },
        {
            "model": MODEL_LABELS["UNET"],
            "input_at_test": "u0, then preceding prediction",
            "output_per_call": "one next field",
            "rollout": "autoregressive",
            "parameters": PARAMETER_COUNTS["UNET"],
        },
    ]
)
display(protocol_table)
protocol_table.to_csv(
    RESULT_ROOT / "baseline_protocol_and_parameters.csv",
    index=False,
)
with open(
    RESULT_ROOT / "baseline_protocol_and_parameters.tex", "w"
) as stream:
    stream.write(protocol_table.to_latex(index=False))



## Optional: disconnect the Colab runtime after every requested run completes



In [ ]:
# Uncomment only after verifying that all requested CSV/JSON/checkpoint files
# have been written to Drive.
#
from google.colab import runtime
runtime.unassign()
